In [3]:
import tensorflow as tf
import tensorflow.contrib.learn as skflow
from sklearn import datasets, metrics
from sklearn.metrics import classification_report,confusion_matrix
from matplotlib import cm
from matplotlib import gridspec
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
import tensorflow as tf

In [41]:
data = pd.read_csv("geo2r.csv")
data = data.drop(["ID","Gene.title","Gene.symbol"],1)
data.describe

<bound method NDFrame.describe of           adj.P.Val       P.Value          t          B         logFC
0      7.830000e-16  1.430000e-20 -11.700000  35.660134 -3.377044e+00
1      2.670000e-14  9.780000e-19 -10.900000  31.668004 -1.458428e+00
2      2.790000e-14  1.530000e-18 -10.800000  31.243509 -2.939485e+00
3      4.060000e-13  2.970000e-17 -10.200000  28.430316 -2.052865e+00
4      4.870000e-12  4.450000e-16  -9.680000  25.859058 -2.604469e+00
5      7.060000e-12  7.740000e-16  -9.570000  25.332792 -2.976581e+00
6      9.250000e-12  1.210000e-15  -9.480000  24.905488 -1.567918e+00
7      9.250000e-12  1.350000e-15  -9.460000  24.801252 -2.426191e+00
8      1.160000e-11  1.920000e-15  -9.390000  24.471478 -2.490702e+00
9      4.550000e-11  8.890000e-15  -9.090000  23.009800 -3.129912e+00
10     4.550000e-11  9.710000e-15  -9.070000  22.926076 -2.675299e+00
11     4.550000e-11  9.980000e-15  -9.060000  22.900012 -2.545085e+00
12     8.210000e-11  2.000000e-14  -8.930000  22.240042 

In [18]:
# Define the input feature: total_rooms.
my_feature = data[["P.Value"]]

# Configure a numeric feature column for total_rooms.
feature_columns = [tf.feature_column.numeric_column("P.Value")]

#hape of logFC data is a one-dimensional array. 
#This is the default shape for numeric_column, so we don't have to pass it as an argument.

In [35]:
# Define the label
targets = data["logFC"]
#features = data["P.Value"]

In [36]:
#Configure the LinearRegressor
#training l using the GradientDescentOptimizer, which implements Mini-Batch Stochastic Gradient Descent (SGD). 
#The learning_rate argument controls the size of the gradient step.
#also apply gradient clipping to our optimizer via clip_gradients_by_norm. 
#Gradient clipping ensures the magnitude of the gradients do not become too large during training, which can cause gradient descent to fail.
# Use gradient descent as the optimizer for training the model.

my_optimizer=tf.train.GradientDescentOptimizer(learning_rate=0.0000001)
my_optimizer = tf.contrib.estimator.clip_gradients_by_norm(my_optimizer, 5.0)

# Configure the linear regression model with our feature columns and optimizer.
# Set a learning rate of 0.0000001 for Gradient Descent.
linear_regressor = tf.estimator.LinearRegressor(
    feature_columns=feature_columns,
    optimizer=my_optimizer
)

INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': 'C:\\Users\\Hannah\\AppData\\Local\\Temp\\tmplysk7atd', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_service': None, '_cluster_spec': <tensorflow.python.training.server_lib.ClusterSpec object at 0x0000022B12B20518>, '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [37]:
#Define the Input Function:  instructs TensorFlow how to preprocess the data & how to batch, shuffle, and repeat it during model training.
#convert pandas feature data into a dict of NumPy arrays. Then use the TensorFlow Dataset API to construct a dataset object from our data, 
#and then break our data into batches of batch_size, to be repeated for the specified number of epochs (num_epochs)
#NOTE: When the default value of num_epochs=None is passed to repeat(), the input data will be repeated indefinitely.
#if shuffle is set to True, we'll shuffle the data so that it's passed to the model randomly during training. The buffer_size argument specifies the size of the dataset from which shuffle will randomly sample.
#Finally, our input function constructs an iterator for the dataset and returns the next batch of data to the LinearRegressor.

def my_input_fn(features, targets, batch_size=1, shuffle=True, num_epochs=None):
    """Trains a linear regression model of one feature.
  
    Args:
      features: pandas DataFrame of features
      targets: pandas DataFrame of targets
      batch_size: Size of batches to be passed to the model
      shuffle: True or False. Whether to shuffle the data.
      num_epochs: Number of epochs for which data should be repeated. None = repeat indefinitely
    Returns:
      Tuple of (features, labels) for next data batch
    """
  
    # Convert pandas data into a dict of np arrays.
    features = {key:np.array(value) for key,value in dict(features).items()}                                           
 
    # Construct a dataset, and configure batching/repeating.
    ds = tf.data.Dataset.from_tensor_slices((features,targets)) # warning: 2GB limit
    ds = ds.batch(batch_size).repeat(num_epochs)
    
    # Shuffle the data, if specified.
    if shuffle:
        ds = ds.shuffle(buffer_size=10000)
    
    # Return the next batch of data.
    features, labels = ds.make_one_shot_iterator().get_next()
    return features, labels

In [38]:
# Train the Model call train() on our linear_regressor to train the model. 
#Wrap my_input_fn in a lambda so we can pass in my_feature and target as arguments. Training for 100 steps to start
_ = linear_regressor.train(
    input_fn = lambda:my_input_fn(features, targets),
    steps=100
)

IndexError: list index out of range

In [ ]:
# Create an input function for predictions.
# Note: Since we're making just one prediction for each example, we don't 
# need to repeat or shuffle the data here.
prediction_input_fn =lambda: my_input_fn(my_feature, targets, num_epochs=1, shuffle=False)

# Call predict() on the linear_regressor to make predictions.
predictions = linear_regressor.predict(input_fn=prediction_input_fn)

# Format predictions as a NumPy array, so we can calculate error metrics.
predictions = np.array([item['predictions'][0] for item in predictions])

# Print Mean Squared Error and Root Mean Squared Error.
mean_squared_error = metrics.mean_squared_error(predictions, targets)
root_mean_squared_error = math.sqrt(mean_squared_error)